[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 中级：实现 BatchNorm

请实现 **批标准化（Batch Normalization）**，同时支持 **训练模式** 和 **推理模式**。

在**训练模式**下，使用**当前批次统计量**并更新运行估计值：

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

其中 $\mu_B$ 和 $\sigma_B^2$ 是 **沿着批次维度（dim=0）** 计算得到的均值和方差。

在**推理模式**下，使用传入的**运行均值/方差**，而非当前批次的统计量。

### 函数签名
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — 对每个特征，在整个批次样本上进行归一化
    # running_mean, running_var: 训练时原地更新；推理时直接使用
```

### 要求
- **禁止**使用 `F.batch_norm`、`nn.BatchNorm1d` 等现成函数
- 批次均值和方差沿 `dim=0` 计算，使用 `unbiased=False`
- 运行统计量的更新方式与 PyTorch 一致：`running = (1 - momentum) * running + momentum * batch_stat`
- 当 `training=False` 时，必须使用 `running_mean` / `running_var` 进行归一化
- 必须支持对 `x`、`gamma`、`beta` 的自动求导（running statistics 视为 buffer，不需要梯度）

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

批归一化是对第 0 维度进行归一化计算

分训练和推理两部分，训练部分需要更新 running_mean 和 running_var, 以得到累积的平均值和方差。

.copy_() 是一个就地操作（In-place operation），它的核心功能是：将一个张量（Tensor）的内容复制到另一个现有的张量中，自动广播。

1. 按 batch 维度计算平均值
2. 按 batch 维度计算方差
3. 计算 x^ , x_hat 帽子
4. 根据动量因子更新 running_mean 和 running_var
5. 缩放和移位

In [ ]:
# ✏️ 在这里实现你的代码

def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
):
    '''
    参数说明:
    x: 输入张量，形状为 (N, D)。其中 N 是批次大小 (Batch Size)，D 是特征维度 (Feature Dimension)。
    gamma: 可学习的缩放参数 (Scale)，形状为 (D,)。
    beta: 可学习的平移参数 (Shift)，形状为 (D,)。
    running_mean: 训练过程中累积的均值估计 (Running Mean)，形状为 (D,)。
    running_var: 训练过程中累积的方差估计 (Running Variance)，形状为 (D,)。
    eps: 防止分母为零的微小值 (Epsilon)，通常取 1e-5。
    momentum: 运行统计量更新时的动量因子，用于平滑 running_mean/var 的更新。
    training: 布尔值。True 表示训练模式（使用当前 Batch 统计量），False 表示推理模式（使用 Running 统计量）。
    '''
    # pass  # 请替换此处的 pass
    # x shape: (N, D)
    
    if training:
        # 1. 计算当前小批次 (mini-batch) 的统计量
        # 沿着批次维度 (dim=0) 计算，保持维度以便广播
        batch_mean = x.mean(dim=0)
        # 注意：PyTorch BN 在计算方差用于归一化时，默认使用有偏估计 (unbiased=False)
        batch_var = x.var(dim=0, unbiased=False)
        
        # 2. 标准化当前批次
        # 公式: x_hat = (x - mu) / sqrt(var + eps)
        x_hat = (x - batch_mean) / torch.sqrt(batch_var + eps)
        
        # 3. 更新运行统计量 (Running Statistics)
        # 使用原地操作 (inplace) 更新，因为 running_mean/var 是传入的 buffer
        # 公式: running = (1 - momentum) * running + momentum * batch_stat
        running_mean.copy_((1 - momentum) * running_mean + momentum * batch_mean)
        running_var.copy_((1 - momentum) * running_var + momentum * batch_var)
    else:
        # 4. 推理模式：直接使用训练期间累积的 running 统计量
        x_hat = (x - running_mean) / torch.sqrt(running_var + eps)
    
    # 5. 缩放和移位 (Affines: Scale and Shift)
    # y = gamma * x_hat + beta
    out = gamma * x_hat + beta
    
    return out

In [ ]:
# 🧪 调试测试
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# 运行统计量通常与特征维度一致，且在同一设备上
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# 训练模式：使用批次统计量，并更新 running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[训练] 输出形状:", out_train.shape)
print("[训练] 各列均值:", out_train.mean(dim=0))   # 应该接近 0
print("[训练] 各列标准差:", out_train.std(dim=0))    # 应该接近 1
print("更新后的 running_mean:", running_mean)
print("更新后的 running_var:", running_var)

# 推理模式：仅使用 running_mean / running_var
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[推理] 输出形状:", out_eval.shape)

In [ ]:
# ✅ 提交验证
from torch_judge import check
check("batchnorm")